In [ ]:
# from peft import LoraConfig
# from trl import SFTConfig, SFTTrainer
# from datasets import load_dataset
# from accelerate import Accelerator
# from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


# accelerator = Accelerator()

# model = AutoModelForCausalLM.from_pretrained(
#     "TheBloke/vicuna-7B-1.1-HF",
#     device_map="auto",
#     load_in_8bit=True,
# )

# peft_config = LoraConfig(
#     r=32,
#     lora_alpha=16,
#     lora_dropout=0.05,
#     bias="none",
#     task_type="CAUSAL_LM",
#     target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
# )

# training_args = SFTConfig(
#     output_dir="output",
#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=16,
#     max_steps=1000,
#     learning_rate=2e-4,
#     logging_steps=10,
#     save_strategy="steps",
#     save_steps=100,
#     save_total_limit=2,
#     fp16=True,
#     optim="adamw_torch",
# )

# trainer = SFTTrainer(
#     model= model,
#     args= training_args,
#     # train_dataset= dataset,
#     peft_config= peft_config,  # Pass config here
# )

# trainer.train()

ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [7]:
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # ungated; swap in any causal LM

# --- data: build a single "text" column the trainer can read ---
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes", split="train[:3000]")
labels = {0: "negative", 1: "positive"}
ds = ds.map(lambda x: {"text": f"Review: {x['text']}\nSentiment: {labels[x['label']]}"},
            remove_columns=ds.column_names)

# --- tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# --- 4-bit model (the Q in QLoRA) ---
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb, device_map={"": 0},
)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

args = SFTConfig(
    output_dir="output",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    max_steps=300,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="steps", save_steps=100, save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    max_length=256,        # called max_seq_length in older TRL versions
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds,
    processing_class=tokenizer,   # called "tokenizer" in older TRL versions
    peft_config=peft_config,
)
trainer.train()

Map: 100%|██████████| 3000/3000 [00:00<00:00, 112330.38 examples/s]


ValueError: Using a `device_map`, `tp_plan`, `torch.device` context manager or setting `torch.set_default_device(device)` requires `accelerate`. You can install it with `pip install accelerate`

In [8]:
import sys, accelerate, torch
print(sys.executable)
print(accelerate.__version__)
print(torch.cuda.is_available())

/Users/prachilahane_02/VSCODE/finetuning/VENV/bin/python
1.15.0
False


In [ ]:
# worked in colab why ?? 
# my conclusion is that maybe the accelerate version is different in colab and local machine.